In [11]:
"""
Notebook script (archivo .py) para generar datasets finales para entrenar 4 modelos
(RF, XGBoost, RNN, LSTM) con dos estrategias: por estación (cada CSV) y combinado
(todas las estaciones juntas).

Cómo usarlo:
 - Actualiza la lista `FILE_PATHS` con las rutas de tus CSV (ya hay un ejemplo con tus rutas).
 - Cambia `OUTPUT_DIR` donde quieras que se guarden los datasets finales.
 - Ajusta los parámetros en la sección `CONFIGURACION` más abajo (ventana entrada/salida,
   métodos de imputación, features a usar, etc.).

Este script realiza:
 1. Carga y normalización temporal (index datetime, ordenado, frecuencia horaria).
 2. Creación de features cíclicas (hora, día, semana, mes, año) con seno y coseno.
 3. Codificación circular de Direc. y Angulo (crea sin/cos y borra original).
 4. Imputación por-variable (estrategias configurables).
 5. Construcción de ventanas deslizantes para predecir O3 con horizonte 72h.
 6. Genera los datasets compatibles con cada modelo y los guarda en disco.

No entrena modelos aquí — sólo prepara y guarda los datasets.
"""

import os
from pathlib import Path
from typing import List, Dict, Tuple

import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.impute import KNNImputer
import joblib

# ---------------------- CONFIGURACION (ajustable) ----------------------
# Rutas (por defecto, pongo las tuyas; cámbialas si deseas)
FILE_PATHS = [
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T1_E1_Alicante.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T1_E2_Elda.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T2_E1_Elche.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T2_E2_Elda.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T3_E1_Valencia.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T3_E2_Bunol.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T4_E1_Valencia.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T4_E2_Villar_Arzobispo.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T6_E1_Castellon.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T6_E2_Onda.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T7_E1_Sant_Jordi.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T7_E2_Coratxa.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T7_E3_Zorita.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T8_E1_Sant_Jordi.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T8_E2_Morella.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T8_E3_Zorita.csv",
]

# Directorio donde se guardarán los datasets finales (CAMBIA A TU RUTA)
OUTPUT_DIR = Path(r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/Datasets_finales")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Ventanas: número de horas de entrada y número de horas a predecir (horizonte)
INPUT_HOURS = 72   # uso por defecto: 72h de historial
OUTPUT_HOURS = 72  # horizonte: 72h (3 días)

# Columnas esperadas (adapta si tus CSV usan nombres distintos)
EXPECTED_COLS = [
    'datetime', 'NO', 'NO2', 'NOx', 'O3', 'Veloc.', 'Direc.', 'Temp.', 'R.Sol.',
    'Estacion', 'Transecto', 'Dist.', 'Angulo'
]

# Imputacion por variable (opciones: 'time_interpolate', 'knn', 'ffill', 'median', 'constant', 'circular')
IMPUTE_STRATEGIES = {
    'NO': 'time_interpolate',
    'NO2': 'time_interpolate',
    'NOx': 'time_interpolate',
    'O3': 'time_interpolate',
    'Veloc.': 'time_interpolate',
    'Direc.': 'circular',
    'Temp.': 'time_interpolate',
    'R.Sol.': 'time_interpolate',
    'Estacion': 'ffill',
    'Transecto': 'ffill',
    'Dist.': 'ffill',
    'Angulo': 'circular'
}

# Modelos a preparar
MODELS = ['rf', 'xgb', 'rnn', 'lstm']

# Features usadas (las cíclicas se generan automáticamente)
BASE_FEATURES = ['NO', 'NO2', 'NOx', 'Veloc.', 'Temp.', 'R.Sol.']
CATEGORICAL = ['Estacion', 'Transecto']
CIRCULAR_ORIG = ['Direc.', 'Angulo']
TARGET = 'O3'

# ---------------------- FUNCIONES AUXILIARES ----------------------

def ensure_datetime_index(df: pd.DataFrame, col: str = 'datetime') -> pd.DataFrame:
    df = df.copy()
    if col in df.columns:
        df[col] = pd.to_datetime(df[col])
        df = df.set_index(col)
    else:
        if not isinstance(df.index, pd.DatetimeIndex):
            raise ValueError('No hay columna datetime ni index datetime.')
    df = df.sort_index()
    # reindex to full hourly range to ensure consistent frequency
    full_idx = pd.date_range(df.index.min(), df.index.max(), freq='H')
    df = df.reindex(full_idx)
    return df


def add_cyclical_datetime_features(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    idx = d.index
    hour = idx.hour
    day = idx.day
    week = idx.isocalendar().week.astype(int)
    month = idx.month
    year = idx.year

    # normalized ranges
    d['hour_sin'] = np.sin(2 * np.pi * hour / 24)
    d['hour_cos'] = np.cos(2 * np.pi * hour / 24)
    d['day_sin'] = np.sin(2 * np.pi * (day - 1) / 31)
    d['day_cos'] = np.cos(2 * np.pi * (day - 1) / 31)
    d['week_sin'] = np.sin(2 * np.pi * (week - 1) / 52)
    d['week_cos'] = np.cos(2 * np.pi * (week - 1) / 52)
    d['month_sin'] = np.sin(2 * np.pi * (month - 1) / 12)
    d['month_cos'] = np.cos(2 * np.pi * (month - 1) / 12)
    # para año se normaliza relativo al primer año
    years = year - year.min()
    d['year_sin'] = np.sin(2 * np.pi * years / (year.max() - year.min() + 1))
    d['year_cos'] = np.cos(2 * np.pi * years / (year.max() - year.min() + 1))
    return d


def circular_to_sincos(series: pd.Series, prefix: str) -> pd.DataFrame:
    # convierte grados (0-360) a sin y cos (valores entre -1 y 1)
    radians = np.deg2rad(series)
    return pd.DataFrame({f'{prefix}_sin': np.sin(radians), f'{prefix}_cos': np.cos(radians)}, index=series.index)


def impute_variable(df: pd.DataFrame, col: str, strategy: str, knn_imputer: KNNImputer = None) -> pd.Series:
    s = df[col]
    if strategy == 'time_interpolate':
        # interpolation time-based; limit_direction both; if still NA fill with median
        s_imputed = s.interpolate(method='time', limit_direction='both')
        if s_imputed.isna().any():
            s_imputed = s_imputed.fillna(s_imputed.median())
        return s_imputed
    elif strategy == 'knn':
        if knn_imputer is None:
            raise ValueError('KNN imputer no provisto')
        arr = knn_imputer.fit_transform(df[[col]])
        return pd.Series(arr.ravel(), index=df.index)
    elif strategy == 'ffill':
        return s.fillna(method='ffill').fillna(method='bfill')
    elif strategy == 'median':
        return s.fillna(s.median())
    elif strategy == 'constant':
        return s.fillna(0)
    elif strategy == 'circular':
        # convert to sin/cos, interpolate separately, return DataFrame handled elsewhere
        raise ValueError('La imputación circular debe manejarse fuera de esta función')
    else:
        raise ValueError(f'Estrategia desconocida: {strategy}')


# ---------------------- PROCESAMIENTO DE UN CSV ----------------------

def preprocess_single_csv(path: str,
                          impute_strategies: Dict[str, str] = IMPUTE_STRATEGIES) -> Tuple[pd.DataFrame, Dict]:
    """
    Carga un CSV, genera features cíclicas, codifica circularmente Direc./Angulo y aplica imputación.
    Devuelve el dataframe procesado y metadatos (mappings para categorías).
    """
    df = pd.read_csv(path)
    # validar cols
    # normalizar nombres de columna - quitar espacios finales si los hubiera
    df.columns = [c.strip() for c in df.columns]

    df = ensure_datetime_index(df, 'datetime')

    # generar features temporales cíclicas
    df = add_cyclical_datetime_features(df)

    # Manejar columnas circulares: Direc. y Angulo -> sin/cos
    for circ_col in CIRCULAR_ORIG:
        if circ_col in df.columns:
            sincos = circular_to_sincos(df[circ_col], circ_col.replace('.', '').replace(' ', '_'))
            # concat, pero NO aplicar imputación aún — imputaremos sin/cos con time_interpolate
            df = pd.concat([df, sincos], axis=1)
    # ahora imputación por variable
    # para circularas, imputamos las nuevas columnas con time interpolation
    # preparar knn_imputer si es necesario
    knn_needed = any(v == 'knn' for v in impute_strategies.values())
    knn_imputer = KNNImputer(n_neighbors=5) if knn_needed else None

    for col, strat in impute_strategies.items():
        if col in CIRCULAR_ORIG and strat == 'circular':
            # imputar sin/cos asociados
            pref = col.replace('.', '').replace(' ', '_')
            for comp in [f'{pref}_sin', f'{pref}_cos']:
                if comp in df.columns:
                    df[comp] = df[comp].interpolate(method='time', limit_direction='both')
                    if df[comp].isna().any():
                        df[comp] = df[comp].fillna(df[comp].median())
        else:
            if col in df.columns:
                df[col] = impute_variable(df, col, strat, knn_imputer)
            else:
                # si falta la columna, crear con NaNs y luego imputar
                df[col] = np.nan
                df[col] = impute_variable(df, col, strat, knn_imputer)

    # para variables categóricas: rellenar con forward/back fill si quedaran NaNs
    for cat in CATEGORICAL:
        if cat in df.columns:
            df[cat] = df[cat].fillna(method='ffill').fillna(method='bfill')

    # asegurar que no haya NaNs en las columnas base y target (por seguridad)
    for col in BASE_FEATURES + [TARGET]:
        if col in df.columns and df[col].isna().any():
            df[col] = df[col].fillna(df[col].median())

    # devolver df
    return df, {'columns': df.columns.tolist()}


# ---------------------- CREAR VENTANAS SUPERVISADAS ----------------------

def make_sliding_windows(df: pd.DataFrame,
                         input_hours: int = INPUT_HOURS,
                         output_hours: int = OUTPUT_HOURS,
                         features: List[str] = None,
                         target_col: str = TARGET,
                         model_type: str = 'rf'):
    """
    Crea muestras deslizantes:
      - Para RF/XGB: devuelve X (2D) y y (2D) en formato tabular (cada muestra una fila, y tiene columnas y_1..y_H)
      - Para RNN/LSTM: devuelve arrays X (n, t, f) y y (n, H) donde y es la secuencia de target
    """
    if features is None:
        features = BASE_FEATURES + [c for c in df.columns if c.startswith('hour_') or c.endswith('_sin') or c.endswith('_cos')]
    # asegurarnos de orden estable de features
    features = [f for f in features if f in df.columns]

    X_list = []
    y_list = []
    idxs = []
    total_len = len(df)
    for start in range(0, total_len - input_hours - output_hours + 1):
        end_input = start + input_hours
        end_output = end_input + output_hours
        Xw = df.iloc[start:end_input][features].values
        yw = df.iloc[end_input:end_output][target_col].values
        # si hay NaNs (debería no haber), saltar
        if np.isnan(Xw).any() or np.isnan(yw).any():
            continue
        X_list.append(Xw)
        y_list.append(yw)
        idxs.append(df.index[end_input - 1])

    if model_type in ['rf', 'xgb']:
        # aplanar X a 2D
        X_flat = np.array(X_list)
        n_samples, t, n_feat = X_flat.shape
        X_2d = X_flat.reshape(n_samples, t * n_feat)
        y_2d = np.array(y_list)  # shape (n_samples, output_hours)
        # crear df con columnas y_1..y_H
        y_cols = [f'y_{i+1}' for i in range(y_2d.shape[1])]
        X_df = pd.DataFrame(X_2d)
        y_df = pd.DataFrame(y_2d, columns=y_cols)
        X_df.index = idxs
        y_df.index = idxs
        return X_df, y_df
    else:
        X_arr = np.array(X_list)  # (n, t, f)
        y_arr = np.array(y_list)  # (n, H)
        return X_arr, y_arr


# ---------------------- CREAR DATASETS POR ESTACION Y COMBINADO ----------------------

def prepare_and_save_all(file_paths: List[str], output_dir: Path):
    # almacenar metadatos para codificadores globales usados en datasets combinados
    global_onehot = OneHotEncoder(sparse=False, handle_unknown='ignore')

    # leer y preprocesar todos
    processed = {}
    for p in file_paths:
        name = Path(p).stem
        print(f'Procesando {name}...')
        df, meta = preprocess_single_csv(p)
        processed[name] = df

    # FIT global onehot sobre todas las estaciones/ transectos (para combinado y para que las columnas sean estables)
    all_cats = pd.DataFrame({
        'Estacion': [df['Estacion'].iloc[0] if 'Estacion' in df.columns and not df['Estacion'].isna().all() else name for name, df in processed.items()],
        'Transecto': [df['Transecto'].iloc[0] if 'Transecto' in df.columns and not df['Transecto'].isna().all() else name for name, df in processed.items()]
    })
    global_onehot.fit(all_cats)
    joblib.dump(global_onehot, output_dir / 'global_onehot_encoder.joblib')

    # 1) Dataset por estación (4 por estación)
    for name, df in processed.items():
        station_dir = output_dir / 'per_station' / name
        station_dir.mkdir(parents=True, exist_ok=True)
        # --- RF/XGB: aplicar one-hot a columnas categóricas (per-estacion no es muy necesario, pero mantengo convención)
        df_tab = df.copy()
        # Si existen categóricas, usar pd.get_dummies para esta estación
        cats = [c for c in CATEGORICAL if c in df_tab.columns]
        if cats:
            df_tab = pd.get_dummies(df_tab, columns=cats, dummy_na=False)

        X_rf, y_rf = make_sliding_windows(df_tab, model_type='rf')
        X_rf.to_csv(station_dir / 'rf_X.csv', index=True)
        y_rf.to_csv(station_dir / 'rf_y.csv', index=True)

        X_xgb, y_xgb = make_sliding_windows(df_tab, model_type='xgb')
        X_xgb.to_csv(station_dir / 'xgb_X.csv', index=True)
        y_xgb.to_csv(station_dir / 'xgb_y.csv', index=True)

        # --- RNN/LSTM: crear arrays 3D y guardarlos como .npz
        # Para RNN/LSTM es recomendable usar label encoding para Estacion/Transecto si se usan
        df_seq = df.copy()
        encoders = {}
        for cat in CATEGORICAL:
            if cat in df_seq.columns:
                le = LabelEncoder()
                df_seq[cat + '_id'] = le.fit_transform(df_seq[cat].astype(str))
                encoders[cat] = le
        joblib.dump(encoders, station_dir / 'label_encoders.joblib')

        # features para secuencia (incluimos las ids si existen y todas las sin/cos)
        seq_features = [f for f in df_seq.columns if (
            f in BASE_FEATURES or f.endswith('_sin') or f.endswith('_cos') or f.endswith('_id') or f.startswith('hour_') or f.startswith('day_') or f.startswith('week_') or f.startswith('month_') or f.startswith('year_')
        )]
        X_rnn, y_rnn = make_sliding_windows(df_seq, features=seq_features, model_type='rnn')
        np.savez_compressed(station_dir / 'rnn_data.npz', X=X_rnn, y=y_rnn)
        # copiar mismo arrays para LSTM (mismo formato)
        np.savez_compressed(station_dir / 'lstm_data.npz', X=X_rnn, y=y_rnn)

    # 2) Dataset combinado (todas las estaciones en uno)
    combined = pd.concat(processed.values(), keys=processed.keys(), names=['station', 'datetime'])
    # quitar el multiindex a nivel datetime y mantener station como columna
    combined = combined.reset_index(level=0).rename(columns={'station': 'Estacion_source'})
    # recomponer index datetime
    combined.index = combined.index.droplevel(0)

    combined_dir = output_dir / 'combined'
    combined_dir.mkdir(parents=True, exist_ok=True)

    # Aplicar global one-hot (usamos el encoder global que guardamos)
    # Pero primero rellenar Estacion y Transecto si faltan
    for cat in CATEGORICAL:
        if cat not in combined.columns:
            combined[cat] = combined['Estacion_source']

    # One-hot con columnas estables
    oh = joblib.load(output_dir / 'global_onehot_encoder.joblib')
    cat_df = combined[['Estacion', 'Transecto']].fillna('unknown').astype(str)
    oh_arr = oh.transform(cat_df)
    oh_cols = oh.get_feature_names_out(['Estacion', 'Transecto'])
    oh_df = pd.DataFrame(oh_arr, index=combined.index, columns=oh_cols)
    combined_tab = pd.concat([combined.drop(columns=CATEGORICAL), oh_df], axis=1)

    # crear datasets combinados para RF/XGB
    X_rf_c, y_rf_c = make_sliding_windows(combined_tab, model_type='rf')
    pd.DataFrame(X_rf_c).to_csv(combined_dir / 'rf_X.csv', index=True)
    pd.DataFrame(y_rf_c).to_csv(combined_dir / 'rf_y.csv', index=True)

    X_xgb_c, y_xgb_c = make_sliding_windows(combined_tab, model_type='xgb')
    pd.DataFrame(X_xgb_c).to_csv(combined_dir / 'xgb_X.csv', index=True)
    pd.DataFrame(y_xgb_c).to_csv(combined_dir / 'xgb_y.csv', index=True)

    # Para RNN/LSTM en combinado: usar label encoding global para Estacion/Transecto
    # crear label encoder global
    enc_global = {}
    for cat in CATEGORICAL:
        le = LabelEncoder()
        combined[cat + '_id'] = le.fit_transform(combined[cat].astype(str))
        enc_global[cat] = le
    joblib.dump(enc_global, combined_dir / 'global_label_encoders.joblib')

    seq_features_comb = [f for f in combined.columns if (
        f in BASE_FEATURES or f.endswith('_sin') or f.endswith('_cos') or f.endswith('_id') or f.startswith('hour_') or f.startswith('day_') or f.startswith('week_') or f.startswith('month_') or f.startswith('year_')
    )]
    X_rnn_c, y_rnn_c = make_sliding_windows(combined, features=seq_features_comb, model_type='rnn')
    np.savez_compressed(combined_dir / 'rnn_data.npz', X=X_rnn_c, y=y_rnn_c)
    np.savez_compressed(combined_dir / 'lstm_data.npz', X=X_rnn_c, y=y_rnn_c)

    print('Datasets generados y guardados en:', output_dir)


if __name__ == '__main__':
    prepare_and_save_all(FILE_PATHS, OUTPUT_DIR)

# Fin del script


TypeError: OneHotEncoder.__init__() got an unexpected keyword argument 'sparse'

In [13]:
"""
Notebook script (archivo .py) para generar datasets finales para entrenar 4 modelos
(RF, XGBoost, RNN, LSTM) con dos estrategias: por estación (cada CSV) y combinado
(todas las estaciones juntas).

VERSIÓN ACTUALIZADA: este script implementa una imputación adaptativa "mejor método"
por variable en función del porcentaje de datos faltantes y la naturaleza de la variable
(continua, circular o categórica). Las reglas por defecto son:
  - < 5% missing -> interpolación temporal (preserva continuidad y estacionalidad)
  - 5% - 30% -> KNN Imputer (usa vecinos temporales/espaciales)
  - 30% - 70% -> Iterative Imputer (model-based, multivariado)
  - > 70% -> Relleno con mediana (o constante) — cuando hay poca información

Para variables circulares (Direc., Angulo) se trabaja con sus componentes sin/cos y
se imputan estas componentes por interpolación temporal siempre que sea posible.

Cómo usarlo:
 - Actualiza la lista `FILE_PATHS` con las rutas de tus CSV (ya hay un ejemplo con tus rutas).
 - Cambia `OUTPUT_DIR` donde quieras que se guarden los datasets finales.
 - Ajusta parámetros en la sección `CONFIGURACION` si quieres anular la elección automática.

Este script realiza:
 1. Carga y normalización temporal (index datetime, ordenado, frecuencia horaria).
 2. Creación de features cíclicas (hora, día, semana, mes, año) con seno y coseno.
 3. Codificación circular de Direc. y Angulo (crea sin/cos y borra original si quieres).
 4. Imputación adaptativa por variable (KNN, IterativeImputer, interpolación, mediana).
 5. Construcción de ventanas deslizantes para predecir O3 con horizonte 72h.
 6. Genera los datasets compatibles con cada modelo y los guarda en disco.

No entrena modelos aquí — sólo prepara y guarda los datasets.
"""

import os
from pathlib import Path
from typing import List, Dict, Tuple

import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
import joblib

# ---------------------- CONFIGURACION (ajustable) ----------------------
# Rutas (por defecto, pongo las tuyas; cámbialas si deseas)
FILE_PATHS = [
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T1_E1_Alicante.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T1_E2_Elda.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T2_E1_Elche.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T2_E2_Elda.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T3_E1_Valencia.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T3_E2_Bunol.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T4_E1_Valencia.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T4_E2_Villar_Arzobispo.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T6_E1_Castellon.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T6_E2_Onda.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T7_E1_Sant_Jordi.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T7_E2_Coratxa.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T7_E3_Zorita.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T8_E1_Sant_Jordi.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T8_E2_Morella.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T8_E3_Zorita.csv",
]

# Directorio donde se guardarán los datasets finales (CAMBIA A TU RUTA)
OUTPUT_DIR = Path(r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datasets_finales")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Ventanas: número de horas de entrada y número de horas a predecir (horizonte)
INPUT_HOURS = 72   # uso por defecto: 72h de historial
OUTPUT_HOURS = 72  # horizonte: 72h (3 días)

# Columnas esperadas (adapta si tus CSV usan nombres distintos)
EXPECTED_COLS = [
    'datetime', 'NO', 'NO2', 'NOx', 'O3', 'Veloc.', 'Direc.', 'Temp.', 'R.Sol.',
    'Estacion', 'Transecto', 'Dist.', 'Angulo'
]

# MODELO: reglas automáticas de selección de imputación según porcentaje de missing
# Puedes sobreescribir por variable en IMPUTE_PREFS
MISSING_THRESHOLDS = {
    'low': 0.05,    # <5%
    'medium': 0.30, # 5%-30%
    'high': 0.70     # 30%-70%
}

# Preferencias por variable (si quieres fijar manualmente, pon estrategia: 'time_interpolate','knn','iterative','median','ffill')
# Por defecto vacío — el script seleccionará automáticamente.
IMPUTE_PREFS: Dict[str, str] = {
    # ejemplo: 'NO2': 'knn'
}

# Modelos a preparar
MODELS = ['rf', 'xgb', 'rnn', 'lstm']

# Features usadas (las cíclicas se generan automáticamente)
BASE_FEATURES = ['NO', 'NO2', 'NOx', 'Veloc.', 'Temp.', 'R.Sol.']
CATEGORICAL = ['Estacion', 'Transecto']
CIRCULAR_ORIG = ['Direc.', 'Angulo']
TARGET = 'O3'

# ---------------------- FUNCIONES AUXILIARES ----------------------

def ensure_datetime_index(df: pd.DataFrame, col: str = 'datetime') -> pd.DataFrame:
    df = df.copy()
    if col in df.columns:
        df[col] = pd.to_datetime(df[col])
        df = df.set_index(col)
    else:
        if not isinstance(df.index, pd.DatetimeIndex):
            raise ValueError('No hay columna datetime ni index datetime.')
    df = df.sort_index()
    # reindex to full hourly range to ensure consistent frequency
    full_idx = pd.date_range(df.index.min(), df.index.max(), freq='H')
    df = df.reindex(full_idx)
    return df


def add_cyclical_datetime_features(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    idx = d.index
    hour = idx.hour
    day = idx.day
    week = idx.isocalendar().week.astype(int)
    month = idx.month
    year = idx.year

    # normalized ranges
    d['hour_sin'] = np.sin(2 * np.pi * hour / 24)
    d['hour_cos'] = np.cos(2 * np.pi * hour / 24)
    d['day_sin'] = np.sin(2 * np.pi * (day - 1) / 31)
    d['day_cos'] = np.cos(2 * np.pi * (day - 1) / 31)
    d['week_sin'] = np.sin(2 * np.pi * (week - 1) / 52)
    d['week_cos'] = np.cos(2 * np.pi * (week - 1) / 52)
    d['month_sin'] = np.sin(2 * np.pi * (month - 1) / 12)
    d['month_cos'] = np.cos(2 * np.pi * (month - 1) / 12)
    # para año se normaliza relativo al primer año
    years = year - year.min()
    d['year_sin'] = np.sin(2 * np.pi * years / (year.max() - year.min() + 1))
    d['year_cos'] = np.cos(2 * np.pi * years / (year.max() - year.min() + 1))
    return d


def circular_to_sincos(series: pd.Series, prefix: str) -> pd.DataFrame:
    # convierte grados (0-360) a sin y cos (valores entre -1 y 1)
    radians = np.deg2rad(series)
    return pd.DataFrame({f'{prefix}_sin': np.sin(radians), f'{prefix}_cos': np.cos(radians)}, index=series.index)


def choose_imputation_strategy(series: pd.Series, var_name: str) -> str:
    """Elige la estrategia óptima según el porcentaje de datos faltantes y tipo de variable.
    Devuelve: 'time_interpolate','knn','iterative','median','ffill'
    """
    # override manual
    if var_name in IMPUTE_PREFS:
        return IMPUTE_PREFS[var_name]

    n = len(series)
    missing = series.isna().sum()
    frac = missing / n if n > 0 else 1.0

    # if variable es categórica
    if var_name in CATEGORICAL:
        return 'ffill'
    # if circular (Direc./Angulo) -> trataremos sus sin/cos por interpolación
    if var_name in CIRCULAR_ORIG:
        return 'circular'

    # reglas basadas en thresholds
    if frac < MISSING_THRESHOLDS['low']:
        return 'time_interpolate'
    if frac < MISSING_THRESHOLDS['medium']:
        return 'knn'
    if frac < MISSING_THRESHOLDS['high']:
        return 'iterative'
    return 'median'


def impute_variable_adaptive(df: pd.DataFrame, col: str) -> pd.Series:
    """Imputa una columna usando la estrategia seleccionada automáticamente.
    """
    strat = choose_imputation_strategy(df[col], col)
    if strat == 'time_interpolate':
        s_imputed = df[col].interpolate(method='time', limit_direction='both')
        if s_imputed.isna().any():
            s_imputed = s_imputed.fillna(s_imputed.median())
        return s_imputed
    elif strat == 'knn':
        # KNN sobre columnas numéricas: usamos las columnas base más la propia
        numeric_cols = [c for c in df.columns if df[c].dtype.kind in 'bifc']
        knn = KNNImputer(n_neighbors=5)
        arr = knn.fit_transform(df[numeric_cols])
        res = pd.DataFrame(arr, index=df.index, columns=numeric_cols)[col]
        return res
    elif strat == 'iterative':
        numeric_cols = [c for c in df.columns if df[c].dtype.kind in 'bifc']
        imp = IterativeImputer(max_iter=10, random_state=0)
        arr = imp.fit_transform(df[numeric_cols])
        res = pd.DataFrame(arr, index=df.index, columns=numeric_cols)[col]
        return res
    elif strat == 'median':
        return df[col].fillna(df[col].median())
    elif strat == 'ffill':
        return df[col].fillna(method='ffill').fillna(method='bfill')
    elif strat == 'circular':
        raise ValueError('La imputación circular debe manejarse fuera de esta función sobre sin/cos')
    else:
        raise ValueError(f'Estrategia desconocida: {strat}')


# ---------------------- PROCESAMIENTO DE UN CSV ----------------------

def preprocess_single_csv(path: str) -> Tuple[pd.DataFrame, Dict]:
    """
    Carga un CSV, genera features cíclicas, codifica circularmente Direc./Angulo y aplica imputación adaptativa.
    Devuelve el dataframe procesado y metadatos (mappings para categorías).
    """
    df = pd.read_csv(path)
    # normalizar nombres de columna - quitar espacios finales si los hubiera
    df.columns = [c.strip() for c in df.columns]

    df = ensure_datetime_index(df, 'datetime')

    # generar features temporales cíclicas
    df = add_cyclical_datetime_features(df)

    # Manejar columnas circulares: Direc. y Angulo -> sin/cos
    for circ_col in CIRCULAR_ORIG:
        if circ_col in df.columns:
            sincos = circular_to_sincos(df[circ_col], circ_col.replace('.', '').replace(' ', '_'))
            df = pd.concat([df, sincos], axis=1)

    # imputar componentes circulares primero (si corresponde)
    for circ_col in CIRCULAR_ORIG:
        pref = circ_col.replace('.', '').replace(' ', '_')
        sin_col = f'{pref}_sin'
        cos_col = f'{pref}_cos'
        if sin_col in df.columns and cos_col in df.columns:
            # preferimos interpolación temporal en las componentes para mantener continuidad
            df[sin_col] = df[sin_col].interpolate(method='time', limit_direction='both')
            df[cos_col] = df[cos_col].interpolate(method='time', limit_direction='both')
            # fallback a mediana si queda NaN
            if df[sin_col].isna().any():
                df[sin_col] = df[sin_col].fillna(df[sin_col].median())
            if df[cos_col].isna().any():
                df[cos_col] = df[cos_col].fillna(df[cos_col].median())

    # ahora imputación adaptativa para el resto de columnas
    for col in set(list(df.columns)):
        if col in [c.replace('.', '').replace(' ', '_') + suffix for c in CIRCULAR_ORIG for suffix in ['_sin','_cos']]:
            continue  # ya imputadas
        if col in ['hour_sin','hour_cos','day_sin','day_cos','week_sin','week_cos','month_sin','month_cos','year_sin','year_cos']:
            # las features cíclicas no deberían tener NaNs porque se derivan del index
            continue
        # saltar index y no-numéricos si queremos
        if col == 'Estacion' or col == 'Transecto' or col == 'Dist.' or col == 'datetime':
            # imputación simple para categóricas
            if col in df.columns:
                df[col] = df[col].fillna(method='ffill').fillna(method='bfill')
            continue
        # si columna existe y es numérica
        if col in df.columns and df[col].dtype.kind in 'bifc':
            df[col] = impute_variable_adaptive(df, col)
        else:
            # columnas no numéricas: rellenar con forward/back
            if col in df.columns:
                df[col] = df[col].fillna(method='ffill').fillna(method='bfill')

    # asegurar que no haya NaNs en las columnas base y target (por seguridad)
    for col in BASE_FEATURES + [TARGET]:
        if col in df.columns and df[col].isna().any():
            df[col] = df[col].fillna(df[col].median())

    return df, {'columns': df.columns.tolist()}


# ---------------------- CREAR VENTANAS SUPERVISADAS ----------------------

def make_sliding_windows(df: pd.DataFrame,
                         input_hours: int = INPUT_HOURS,
                         output_hours: int = OUTPUT_HOURS,
                         features: List[str] = None,
                         target_col: str = TARGET,
                         model_type: str = 'rf'):
    """
    Crea muestras deslizantes:
      - Para RF/XGB: devuelve X (2D) y y (2D) en formato tabular (cada muestra una fila, y tiene columnas y_1..y_H)
      - Para RNN/LSTM: devuelve arrays X (n, t, f) y y (n, H) donde y es la secuencia de target
    """
    if features is None:
        features = BASE_FEATURES + [c for c in df.columns if c.startswith('hour_') or c.endswith('_sin') or c.endswith('_cos')]
    # asegurarnos de orden estable de features
    features = [f for f in features if f in df.columns]

    X_list = []
    y_list = []
    idxs = []
    total_len = len(df)
    for start in range(0, total_len - input_hours - output_hours + 1):
        end_input = start + input_hours
        end_output = end_input + output_hours
        Xw = df.iloc[start:end_input][features].values
        yw = df.iloc[end_input:end_output][target_col].values
        # si hay NaNs (debería no haber), saltar
        if np.isnan(Xw).any() or np.isnan(yw).any():
            continue
        X_list.append(Xw)
        y_list.append(yw)
        idxs.append(df.index[end_input - 1])

    if model_type in ['rf', 'xgb']:
        # aplanar X a 2D
        X_flat = np.array(X_list)
        if X_flat.size == 0:
            return pd.DataFrame(), pd.DataFrame()
        n_samples, t, n_feat = X_flat.shape
        X_2d = X_flat.reshape(n_samples, t * n_feat)
        y_2d = np.array(y_list)  # shape (n_samples, output_hours)
        # crear df con columnas y_1..y_H
        y_cols = [f'y_{i+1}' for i in range(y_2d.shape[1])]
        X_df = pd.DataFrame(X_2d)
        y_df = pd.DataFrame(y_2d, columns=y_cols)
        X_df.index = idxs
        y_df.index = idxs
        return X_df, y_df
    else:
        if len(X_list) == 0:
            return np.empty((0, input_hours, len(features))), np.empty((0, output_hours))
        X_arr = np.array(X_list)  # (n, t, f)
        y_arr = np.array(y_list)  # (n, H)
        return X_arr, y_arr


# ---------------------- CREAR DATASETS POR ESTACION Y COMBINADO ----------------------

def prepare_and_save_all(file_paths: List[str], output_dir: Path):
    # almacenar metadatos para codificadores globales usados en datasets combinados
    global_onehot = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

    # leer y preprocesar todos
    processed = {}
    for p in file_paths:
        name = Path(p).stem
        print(f'Procesando {name}...')
        df, meta = preprocess_single_csv(p)
        processed[name] = df

    # FIT global onehot sobre todas las estaciones/ transectos (para combinado y para que las columnas sean estables)
    all_cats = pd.DataFrame({
        'Estacion': [df['Estacion'].iloc[0] if 'Estacion' in df.columns and not df['Estacion'].isna().all() else name for name, df in processed.items()],
        'Transecto': [df['Transecto'].iloc[0] if 'Transecto' in df.columns and not df['Transecto'].isna().all() else name for name, df in processed.items()]
    })
    global_onehot.fit(all_cats)
    joblib.dump(global_onehot, output_dir / 'global_onehot_encoder.joblib')

    # 1) Dataset por estación (4 por estación)
    for name, df in processed.items():
        station_dir = output_dir / 'per_station' / name
        station_dir.mkdir(parents=True, exist_ok=True)
        # --- RF/XGB: aplicar one-hot a columnas categóricas (per-estacion no es muy necesario, pero mantengo convención)
        df_tab = df.copy()
        # Si existen categóricas, usar pd.get_dummies para esta estación
        cats = [c for c in CATEGORICAL if c in df_tab.columns]
        if cats:
            df_tab = pd.get_dummies(df_tab, columns=cats, dummy_na=False)

        X_rf, y_rf = make_sliding_windows(df_tab, model_type='rf')
        if not X_rf.empty:
            X_rf.to_csv(station_dir / 'rf_X.csv', index=True)
            y_rf.to_csv(station_dir / 'rf_y.csv', index=True)
        else:
            print(f'Advertencia: no se generaron ventanas RF para {name} (poca longitud)')

        X_xgb, y_xgb = make_sliding_windows(df_tab, model_type='xgb')
        if not X_xgb.empty:
            X_xgb.to_csv(station_dir / 'xgb_X.csv', index=True)
            y_xgb.to_csv(station_dir / 'xgb_y.csv', index=True)

        # --- RNN/LSTM: crear arrays 3D y guardarlos como .npz
        df_seq = df.copy()
        encoders = {}
        for cat in CATEGORICAL:
            if cat in df_seq.columns:
                le = LabelEncoder()
                df_seq[cat + '_id'] = le.fit_transform(df_seq[cat].astype(str))
                encoders[cat] = le
        joblib.dump(encoders, station_dir / 'label_encoders.joblib')

        # features para secuencia (incluimos las ids si existen y todas las sin/cos)
        seq_features = [f for f in df_seq.columns if (
            f in BASE_FEATURES or f.endswith('_sin') or f.endswith('_cos') or f.endswith('_id') or f.startswith('hour_') or f.startswith('day_') or f.startswith('week_') or f.startswith('month_') or f.startswith('year_')
        )]
        X_rnn, y_rnn = make_sliding_windows(df_seq, features=seq_features, model_type='rnn')
        np.savez_compressed(station_dir / 'rnn_data.npz', X=X_rnn, y=y_rnn)
        np.savez_compressed(station_dir / 'lstm_data.npz', X=X_rnn, y=y_rnn)

    # 2) Dataset combinado (todas las estaciones en uno)
    combined = pd.concat(processed.values(), keys=processed.keys(), names=['station', 'datetime'])
    # quitar el multiindex a nivel datetime y mantener station como columna
    combined = combined.reset_index(level=0).rename(columns={'station': 'Estacion_source'})
    # recomponer index datetime
    combined.index = combined.index.droplevel(0)

    combined_dir = output_dir / 'combined'
    combined_dir.mkdir(parents=True, exist_ok=True)

    # Aplicar global one-hot (usamos el encoder global que guardamos)
    # Pero primero rellenar Estacion y Transecto si faltan
    for cat in CATEGORICAL:
        if cat not in combined.columns:
            combined[cat] = combined['Estacion_source']

    # One-hot con columnas estables
    oh = joblib.load(output_dir / 'global_onehot_encoder.joblib')
    cat_df = combined[['Estacion', 'Transecto']].fillna('unknown').astype(str)
    oh_arr = oh.transform(cat_df)
    oh_cols = oh.get_feature_names_out(['Estacion', 'Transecto'])
    oh_df = pd.DataFrame(oh_arr, index=combined.index, columns=oh_cols)
    combined_tab = pd.concat([combined.drop(columns=CATEGORICAL), oh_df], axis=1)

    # crear datasets combinados para RF/XGB
    X_rf_c, y_rf_c = make_sliding_windows(combined_tab, model_type='rf')
    if not X_rf_c.empty:
        pd.DataFrame(X_rf_c).to_csv(combined_dir / 'rf_X.csv', index=True)
        pd.DataFrame(y_rf_c).to_csv(combined_dir / 'rf_y.csv', index=True)

    X_xgb_c, y_xgb_c = make_sliding_windows(combined_tab, model_type='xgb')
    if not X_xgb_c.empty:
        pd.DataFrame(X_xgb_c).to_csv(combined_dir / 'xgb_X.csv', index=True)
        pd.DataFrame(y_xgb_c).to_csv(combined_dir / 'xgb_y.csv', index=True)

    # Para RNN/LSTM en combinado: usar label encoding global para Estacion/Transecto
    enc_global = {}
    for cat in CATEGORICAL:
        le = LabelEncoder()
        combined[cat + '_id'] = le.fit_transform(combined[cat].astype(str))
        enc_global[cat] = le
    joblib.dump(enc_global, combined_dir / 'global_label_encoders.joblib')

    seq_features_comb = [f for f in combined.columns if (
        f in BASE_FEATURES or f.endswith('_sin') or f.endswith('_cos') or f.endswith('_id') or f.startswith('hour_') or f.startswith('day_') or f.startswith('week_') or f.startswith('month_') or f.startswith('year_')
    )]
    X_rnn_c, y_rnn_c = make_sliding_windows(combined, features=seq_features_comb, model_type='rnn')
    np.savez_compressed(combined_dir / 'rnn_data.npz', X=X_rnn_c, y=y_rnn_c)
    np.savez_compressed(combined_dir / 'lstm_data.npz', X=X_rnn_c, y=y_rnn_c)

    print('Datasets generados y guardados en:', output_dir)


if __name__ == '__main__':
    prepare_and_save_all(FILE_PATHS, OUTPUT_DIR)

# Fin del script


Procesando T1_E1_Alicante...


/var/folders/85/2m2sr7rx6g5d561zn04dcq000000gn/T/ipykernel_20013/2789370638.py:115: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  full_idx = pd.date_range(df.index.min(), df.index.max(), freq='H')


ValueError: La imputación circular debe manejarse fuera de esta función sobre sin/cos

In [15]:
"""
Notebook script (archivo .py) para generar datasets finales para entrenar 4 modelos
(RF, XGBoost, RNN, LSTM) con dos estrategias: por estación (cada CSV) y combinado
(todas las estaciones juntas).

VERSIÓN ACTUALIZADA: este script implementa una imputación adaptativa "mejor método"
por variable en función del porcentaje de datos faltantes y la naturaleza de la variable
(continua, circular o categórica). Las reglas por defecto son:
  - < 5% missing -> interpolación temporal (preserva continuidad y estacionalidad)
  - 5% - 30% -> KNN Imputer (usa vecinos temporales/espaciales)
  - 30% - 70% -> Iterative Imputer (model-based, multivariado)
  - > 70% -> Relleno con mediana (o constante) — cuando hay poca información

Para variables circulares (Direc., Angulo) se trabaja con sus componentes sin/cos y
se imputan estas componentes por interpolación temporal siempre que sea posible.

Cómo usarlo:
 - Actualiza la lista `FILE_PATHS` con las rutas de tus CSV (ya hay un ejemplo con tus rutas).
 - Cambia `OUTPUT_DIR` donde quieras que se guarden los datasets finales.
 - Ajusta parámetros en la sección `CONFIGURACION` si quieres anular la elección automática.

Este script realiza:
 1. Carga y normalización temporal (index datetime, ordenado, frecuencia horaria).
 2. Creación de features cíclicas (hora, día, semana, mes, año) con seno y coseno.
 3. Codificación circular de Direc. y Angulo (crea sin/cos y borra original si quieres).
 4. Imputación adaptativa por variable (KNN, IterativeImputer, interpolación, mediana).
 5. Construcción de ventanas deslizantes para predecir O3 con horizonte 72h.
 6. Genera los datasets compatibles con cada modelo y los guarda en disco.

No entrena modelos aquí — sólo prepara y guarda los datasets.
"""

import os
from pathlib import Path
from typing import List, Dict, Tuple

import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
import joblib

# ---------------------- CONFIGURACION (ajustable) ----------------------
# Rutas (por defecto, pongo las tuyas; cámbialas si deseas)
FILE_PATHS = [
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T1_E1_Alicante.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T1_E2_Elda.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T2_E1_Elche.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T2_E2_Elda.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T3_E1_Valencia.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T3_E2_Bunol.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T4_E1_Valencia.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T4_E2_Villar_Arzobispo.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T6_E1_Castellon.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T6_E2_Onda.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T7_E1_Sant_Jordi.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T7_E2_Coratxa.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T7_E3_Zorita.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T8_E1_Sant_Jordi.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T8_E2_Morella.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T8_E3_Zorita.csv",
]

# Directorio donde se guardarán los datasets finales (CAMBIA A TU RUTA)
OUTPUT_DIR = Path(r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datasets_finales")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Ventanas: número de horas de entrada y número de horas a predecir (horizonte)
INPUT_HOURS = 72   # uso por defecto: 72h de historial
OUTPUT_HOURS = 72  # horizonte: 72h (3 días)

# Columnas esperadas (adapta si tus CSV usan nombres distintos)
EXPECTED_COLS = [
    'datetime', 'NO', 'NO2', 'NOx', 'O3', 'Veloc.', 'Direc.', 'Temp.', 'R.Sol.',
    'Estacion', 'Transecto', 'Dist.', 'Angulo'
]

# MODELO: reglas automáticas de selección de imputación según porcentaje de missing
# Puedes sobreescribir por variable en IMPUTE_PREFS
MISSING_THRESHOLDS = {
    'low': 0.05,    # <5%
    'medium': 0.30, # 5%-30%
    'high': 0.70     # 30%-70%
}

# Preferencias por variable (si quieres fijar manualmente, pon estrategia: 'time_interpolate','knn','iterative','median','ffill')
# Por defecto vacío — el script seleccionará automáticamente.
IMPUTE_PREFS: Dict[str, str] = {
    # ejemplo: 'NO2': 'knn'
}

# Modelos a preparar
MODELS = ['rf', 'xgb', 'rnn', 'lstm']

# Features usadas (las cíclicas se generan automáticamente)
BASE_FEATURES = ['NO', 'NO2', 'NOx', 'Veloc.', 'Temp.', 'R.Sol.']
CATEGORICAL = ['Estacion', 'Transecto']
CIRCULAR_ORIG = ['Direc.', 'Angulo']
TARGET = 'O3'

# ---------------------- FUNCIONES AUXILIARES ----------------------

def ensure_datetime_index(df: pd.DataFrame, col: str = 'datetime') -> pd.DataFrame:
    df = df.copy()
    if col in df.columns:
        df[col] = pd.to_datetime(df[col])
        df = df.set_index(col)
    else:
        if not isinstance(df.index, pd.DatetimeIndex):
            raise ValueError('No hay columna datetime ni index datetime.')
    df = df.sort_index()
    # reindex to full hourly range to ensure consistent frequency
    full_idx = pd.date_range(df.index.min(), df.index.max(), freq='h')
    df = df.reindex(full_idx)
    return df


def add_cyclical_datetime_features(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    idx = d.index
    hour = idx.hour
    day = idx.day
    week = idx.isocalendar().week.astype(int)
    month = idx.month
    year = idx.year

    # normalized ranges
    d['hour_sin'] = np.sin(2 * np.pi * hour / 24)
    d['hour_cos'] = np.cos(2 * np.pi * hour / 24)
    d['day_sin'] = np.sin(2 * np.pi * (day - 1) / 31)
    d['day_cos'] = np.cos(2 * np.pi * (day - 1) / 31)
    d['week_sin'] = np.sin(2 * np.pi * (week - 1) / 52)
    d['week_cos'] = np.cos(2 * np.pi * (week - 1) / 52)
    d['month_sin'] = np.sin(2 * np.pi * (month - 1) / 12)
    d['month_cos'] = np.cos(2 * np.pi * (month - 1) / 12)
    # para año se normaliza relativo al primer año
    years = year - year.min()
    d['year_sin'] = np.sin(2 * np.pi * years / (year.max() - year.min() + 1))
    d['year_cos'] = np.cos(2 * np.pi * years / (year.max() - year.min() + 1))
    return d


def circular_to_sincos(series: pd.Series, prefix: str) -> pd.DataFrame:
    # convierte grados (0-360) a sin y cos (valores entre -1 y 1)
    radians = np.deg2rad(series)
    return pd.DataFrame({f'{prefix}_sin': np.sin(radians), f'{prefix}_cos': np.cos(radians)}, index=series.index)


def choose_imputation_strategy(series: pd.Series, var_name: str) -> str:
    """Elige la estrategia óptima según el porcentaje de datos faltantes y tipo de variable.
    Devuelve: 'time_interpolate','knn','iterative','median','ffill'
    """
    # override manual
    if var_name in IMPUTE_PREFS:
        return IMPUTE_PREFS[var_name]

    n = len(series)
    missing = series.isna().sum()
    frac = missing / n if n > 0 else 1.0

    # if variable es categórica
    if var_name in CATEGORICAL:
        return 'ffill'
    # if circular (Direc./Angulo) -> trataremos sus sin/cos por interpolación
    if var_name in CIRCULAR_ORIG:
        return 'circular'

    # reglas basadas en thresholds
    if frac < MISSING_THRESHOLDS['low']:
        return 'time_interpolate'
    if frac < MISSING_THRESHOLDS['medium']:
        return 'knn'
    if frac < MISSING_THRESHOLDS['high']:
        return 'iterative'
    return 'median'


def impute_variable_adaptive(df: pd.DataFrame, col: str) -> pd.Series:
    """Imputa una columna usando la estrategia seleccionada automáticamente.
    """
    strat = choose_imputation_strategy(df[col], col)
    if strat == 'time_interpolate':
        s_imputed = df[col].interpolate(method='time', limit_direction='both')
        if s_imputed.isna().any():
            s_imputed = s_imputed.fillna(s_imputed.median())
        return s_imputed
    elif strat == 'knn':
        # KNN sobre columnas numéricas: usamos las columnas base más la propia
        numeric_cols = [c for c in df.columns if df[c].dtype.kind in 'bifc']
        knn = KNNImputer(n_neighbors=5)
        arr = knn.fit_transform(df[numeric_cols])
        res = pd.DataFrame(arr, index=df.index, columns=numeric_cols)[col]
        return res
    elif strat == 'iterative':
        numeric_cols = [c for c in df.columns if df[c].dtype.kind in 'bifc']
        imp = IterativeImputer(max_iter=10, random_state=0)
        arr = imp.fit_transform(df[numeric_cols])
        res = pd.DataFrame(arr, index=df.index, columns=numeric_cols)[col]
        return res
    elif strat == 'median':
        return df[col].fillna(df[col].median())
    elif strat == 'ffill':
        return df[col].fillna(method='ffill').fillna(method='bfill')
    elif strat == 'circular':
        raise ValueError('La imputación circular debe manejarse fuera de esta función sobre sin/cos')
    else:
        raise ValueError(f'Estrategia desconocida: {strat}')


# ---------------------- PROCESAMIENTO DE UN CSV ----------------------

def preprocess_single_csv(path: str) -> Tuple[pd.DataFrame, Dict]:
    """
    Carga un CSV, genera features cíclicas, codifica circularmente Direc./Angulo y aplica imputación adaptativa.
    Devuelve el dataframe procesado y metadatos (mappings para categorías).
    """
    df = pd.read_csv(path)
    # normalizar nombres de columna - quitar espacios finales si los hubiera
    df.columns = [c.strip() for c in df.columns]

    df = ensure_datetime_index(df, 'datetime')

    # generar features temporales cíclicas
    df = add_cyclical_datetime_features(df)

    # Manejar columnas circulares: Direc. y Angulo -> sin/cos
    for circ_col in CIRCULAR_ORIG:
        if circ_col in df.columns:
            sincos = circular_to_sincos(df[circ_col], circ_col.replace('.', '').replace(' ', '_'))
            df = pd.concat([df, sincos], axis=1)

    # imputar componentes circulares primero (si corresponde)
    for circ_col in CIRCULAR_ORIG:
        pref = circ_col.replace('.', '').replace(' ', '_')
        sin_col = f'{pref}_sin'
        cos_col = f'{pref}_cos'
        if sin_col in df.columns and cos_col in df.columns:
            # preferimos interpolación temporal en las componentes para mantener continuidad
            df[sin_col] = df[sin_col].interpolate(method='time', limit_direction='both')
            df[cos_col] = df[cos_col].interpolate(method='time', limit_direction='both')
            # fallback a mediana si queda NaN
            if df[sin_col].isna().any():
                df[sin_col] = df[sin_col].fillna(df[sin_col].median())
            if df[cos_col].isna().any():
                df[cos_col] = df[cos_col].fillna(df[cos_col].median())
    # ahora imputación adaptativa para el resto de columnas
    for col in set(list(df.columns)):
        # Saltar columnas originales circulares (Direc., Angulo) porque ya imputamos sus sin/cos
        if col in CIRCULAR_ORIG:
            continue
        # Saltar las componentes sin/cos asociadas a las variables circulares (ya imputadas)
        if any(col == (c.replace('.', '').replace(' ', '_') + suffix) for c in CIRCULAR_ORIG for suffix in ['_sin', '_cos']):
            continue
        # Las features cíclicas derivadas del índice no deberían tener NaNs
        if col in ['hour_sin', 'hour_cos', 'day_sin', 'day_cos', 'week_sin', 'week_cos', 'month_sin', 'month_cos', 'year_sin', 'year_cos']:
            continue
        # Columnas categóricas simples: rellenar con forward/back fill
        if col in ['Estacion', 'Transecto', 'Dist.', 'datetime']:
            if col in df.columns:
                df[col] = df[col].fillna(method='ffill').fillna(method='bfill')
            continue
        # Si columna existe y es numérica -> imputación adaptativa
        if col in df.columns and df[col].dtype.kind in 'bifc':
            df[col] = impute_variable_adaptive(df, col)
        else:
            # columnas no numéricas: rellenar con forward/back
            if col in df.columns:
                df[col] = df[col].fillna(method='ffill').fillna(method='bfill')

    # asegurar que no haya NaNs en las columnas base y target (por seguridad)
    for col in BASE_FEATURES + [TARGET]:
        if col in df.columns and df[col].isna().any():
            df[col] = df[col].fillna(df[col].median())

    return df, {'columns': df.columns.tolist()}


# ---------------------- CREAR VENTANAS SUPERVISADAS ----------------------

def make_sliding_windows(df: pd.DataFrame,
                         input_hours: int = INPUT_HOURS,
                         output_hours: int = OUTPUT_HOURS,
                         features: List[str] = None,
                         target_col: str = TARGET,
                         model_type: str = 'rf'):
    """
    Crea muestras deslizantes:
      - Para RF/XGB: devuelve X (2D) y y (2D) en formato tabular (cada muestra una fila, y tiene columnas y_1..y_H)
      - Para RNN/LSTM: devuelve arrays X (n, t, f) y y (n, H) donde y es la secuencia de target
    """
    if features is None:
        features = BASE_FEATURES + [c for c in df.columns if c.startswith('hour_') or c.endswith('_sin') or c.endswith('_cos')]
    # asegurarnos de orden estable de features
    features = [f for f in features if f in df.columns]

    X_list = []
    y_list = []
    idxs = []
    total_len = len(df)
    for start in range(0, total_len - input_hours - output_hours + 1):
        end_input = start + input_hours
        end_output = end_input + output_hours
        Xw = df.iloc[start:end_input][features].values
        yw = df.iloc[end_input:end_output][target_col].values
        # si hay NaNs (debería no haber), saltar
        if np.isnan(Xw).any() or np.isnan(yw).any():
            continue
        X_list.append(Xw)
        y_list.append(yw)
        idxs.append(df.index[end_input - 1])

    if model_type in ['rf', 'xgb']:
        # aplanar X a 2D
        X_flat = np.array(X_list)
        if X_flat.size == 0:
            return pd.DataFrame(), pd.DataFrame()
        n_samples, t, n_feat = X_flat.shape
        X_2d = X_flat.reshape(n_samples, t * n_feat)
        y_2d = np.array(y_list)  # shape (n_samples, output_hours)
        # crear df con columnas y_1..y_H
        y_cols = [f'y_{i+1}' for i in range(y_2d.shape[1])]
        X_df = pd.DataFrame(X_2d)
        y_df = pd.DataFrame(y_2d, columns=y_cols)
        X_df.index = idxs
        y_df.index = idxs
        return X_df, y_df
    else:
        if len(X_list) == 0:
            return np.empty((0, input_hours, len(features))), np.empty((0, output_hours))
        X_arr = np.array(X_list)  # (n, t, f)
        y_arr = np.array(y_list)  # (n, H)
        return X_arr, y_arr


# ---------------------- CREAR DATASETS POR ESTACION Y COMBINADO ----------------------

def prepare_and_save_all(file_paths: List[str], output_dir: Path):
    # almacenar metadatos para codificadores globales usados en datasets combinados
    global_onehot = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

    # leer y preprocesar todos
    processed = {}
    for p in file_paths:
        name = Path(p).stem
        print(f'Procesando {name}...')
        df, meta = preprocess_single_csv(p)
        processed[name] = df

    # FIT global onehot sobre todas las estaciones/ transectos (para combinado y para que las columnas sean estables)
    all_cats = pd.DataFrame({
        'Estacion': [df['Estacion'].iloc[0] if 'Estacion' in df.columns and not df['Estacion'].isna().all() else name for name, df in processed.items()],
        'Transecto': [df['Transecto'].iloc[0] if 'Transecto' in df.columns and not df['Transecto'].isna().all() else name for name, df in processed.items()]
    })
    global_onehot.fit(all_cats)
    joblib.dump(global_onehot, output_dir / 'global_onehot_encoder.joblib')

    # 1) Dataset por estación (4 por estación)
    for name, df in processed.items():
        station_dir = output_dir / 'per_station' / name
        station_dir.mkdir(parents=True, exist_ok=True)
        # --- RF/XGB: aplicar one-hot a columnas categóricas (per-estacion no es muy necesario, pero mantengo convención)
        df_tab = df.copy()
        # Si existen categóricas, usar pd.get_dummies para esta estación
        cats = [c for c in CATEGORICAL if c in df_tab.columns]
        if cats:
            df_tab = pd.get_dummies(df_tab, columns=cats, dummy_na=False)

        X_rf, y_rf = make_sliding_windows(df_tab, model_type='rf')
        if not X_rf.empty:
            X_rf.to_csv(station_dir / 'rf_X.csv', index=True)
            y_rf.to_csv(station_dir / 'rf_y.csv', index=True)
        else:
            print(f'Advertencia: no se generaron ventanas RF para {name} (poca longitud)')

        X_xgb, y_xgb = make_sliding_windows(df_tab, model_type='xgb')
        if not X_xgb.empty:
            X_xgb.to_csv(station_dir / 'xgb_X.csv', index=True)
            y_xgb.to_csv(station_dir / 'xgb_y.csv', index=True)

        # --- RNN/LSTM: crear arrays 3D y guardarlos como .npz
        df_seq = df.copy()
        encoders = {}
        for cat in CATEGORICAL:
            if cat in df_seq.columns:
                le = LabelEncoder()
                df_seq[cat + '_id'] = le.fit_transform(df_seq[cat].astype(str))
                encoders[cat] = le
        joblib.dump(encoders, station_dir / 'label_encoders.joblib')

        # features para secuencia (incluimos las ids si existen y todas las sin/cos)
        seq_features = [f for f in df_seq.columns if (
            f in BASE_FEATURES or f.endswith('_sin') or f.endswith('_cos') or f.endswith('_id') or f.startswith('hour_') or f.startswith('day_') or f.startswith('week_') or f.startswith('month_') or f.startswith('year_')
        )]
        X_rnn, y_rnn = make_sliding_windows(df_seq, features=seq_features, model_type='rnn')
        np.savez_compressed(station_dir / 'rnn_data.npz', X=X_rnn, y=y_rnn)
        np.savez_compressed(station_dir / 'lstm_data.npz', X=X_rnn, y=y_rnn)

    # 2) Dataset combinado (todas las estaciones en uno)
    combined = pd.concat(processed.values(), keys=processed.keys(), names=['station', 'datetime'])
    # quitar el multiindex a nivel datetime y mantener station como columna
    combined = combined.reset_index(level=0).rename(columns={'station': 'Estacion_source'})
    # recomponer index datetime
    combined.index = combined.index.droplevel(0)

    combined_dir = output_dir / 'combined'
    combined_dir.mkdir(parents=True, exist_ok=True)

    # Aplicar global one-hot (usamos el encoder global que guardamos)
    # Pero primero rellenar Estacion y Transecto si faltan
    for cat in CATEGORICAL:
        if cat not in combined.columns:
            combined[cat] = combined['Estacion_source']

    # One-hot con columnas estables
    oh = joblib.load(output_dir / 'global_onehot_encoder.joblib')
    cat_df = combined[['Estacion', 'Transecto']].fillna('unknown').astype(str)
    oh_arr = oh.transform(cat_df)
    oh_cols = oh.get_feature_names_out(['Estacion', 'Transecto'])
    oh_df = pd.DataFrame(oh_arr, index=combined.index, columns=oh_cols)
    combined_tab = pd.concat([combined.drop(columns=CATEGORICAL), oh_df], axis=1)

    # crear datasets combinados para RF/XGB
    X_rf_c, y_rf_c = make_sliding_windows(combined_tab, model_type='rf')
    if not X_rf_c.empty:
        pd.DataFrame(X_rf_c).to_csv(combined_dir / 'rf_X.csv', index=True)
        pd.DataFrame(y_rf_c).to_csv(combined_dir / 'rf_y.csv', index=True)

    X_xgb_c, y_xgb_c = make_sliding_windows(combined_tab, model_type='xgb')
    if not X_xgb_c.empty:
        pd.DataFrame(X_xgb_c).to_csv(combined_dir / 'xgb_X.csv', index=True)
        pd.DataFrame(y_xgb_c).to_csv(combined_dir / 'xgb_y.csv', index=True)

    # Para RNN/LSTM en combinado: usar label encoding global para Estacion/Transecto
    enc_global = {}
    for cat in CATEGORICAL:
        le = LabelEncoder()
        combined[cat + '_id'] = le.fit_transform(combined[cat].astype(str))
        enc_global[cat] = le
    joblib.dump(enc_global, combined_dir / 'global_label_encoders.joblib')

    seq_features_comb = [f for f in combined.columns if (
        f in BASE_FEATURES or f.endswith('_sin') or f.endswith('_cos') or f.endswith('_id') or f.startswith('hour_') or f.startswith('day_') or f.startswith('week_') or f.startswith('month_') or f.startswith('year_')
    )]
    X_rnn_c, y_rnn_c = make_sliding_windows(combined, features=seq_features_comb, model_type='rnn')
    np.savez_compressed(combined_dir / 'rnn_data.npz', X=X_rnn_c, y=y_rnn_c)
    np.savez_compressed(combined_dir / 'lstm_data.npz', X=X_rnn_c, y=y_rnn_c)

    print('Datasets generados y guardados en:', output_dir)


if __name__ == '__main__':
    prepare_and_save_all(FILE_PATHS, OUTPUT_DIR)

# Fin del script


Procesando T1_E1_Alicante...


/var/folders/85/2m2sr7rx6g5d561zn04dcq000000gn/T/ipykernel_20013/3124780886.py:262: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[col] = df[col].fillna(method='ffill').fillna(method='bfill')
/var/folders/85/2m2sr7rx6g5d561zn04dcq000000gn/T/ipykernel_20013/3124780886.py:262: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[col] = df[col].fillna(method='ffill').fillna(method='bfill')
/var/folders/85/2m2sr7rx6g5d561zn04dcq000000gn/T/ipykernel_20013/3124780886.py:262: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[col] = df[col].fillna(method='ffill').fillna(method='bfill')


Procesando T1_E2_Elda...


KeyboardInterrupt: 

In [ ]:
"""
Notebook script (archivo .py) para generar datasets finales para entrenar 4 modelos
(RF, XGBoost, RNN, LSTM) con dos estrategias: por estación (cada CSV) y combinado
(todas las estaciones juntas).
(versión corregida)
"""
import os
from pathlib import Path
from typing import List, Dict, Tuple

import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
import joblib

# ---------------------- CONFIGURACION (ajustable) ----------------------
FILE_PATHS = [
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T1_E1_Alicante.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T1_E2_Elda.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T2_E1_Elche.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T2_E2_Elda.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T3_E1_Valencia.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T3_E2_Bunol.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T4_E1_Valencia.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T4_E2_Villar_Arzobispo.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T6_E1_Castellon.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T6_E2_Onda.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T7_E1_Sant_Jordi.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T7_E2_Coratxa.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T7_E3_Zorita.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T8_E1_Sant_Jordi.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T8_E2_Morella.csv",
    r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T8_E3_Zorita.csv",
]

OUTPUT_DIR = Path(r"/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/Datasets_finales")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_HOURS = 72
OUTPUT_HOURS = 72

EXPECTED_COLS = [
    'datetime', 'NO', 'NO2', 'NOx', 'O3', 'Veloc.', 'Direc.', 'Temp.', 'R.Sol.',
    'Estacion', 'Transecto', 'Dist.', 'Angulo'
]

MISSING_THRESHOLDS = {
    'low': 0.05,
    'medium': 0.30,
    'high': 0.70
}

IMPUTE_PREFS: Dict[str, str] = {}

MODELS = ['rf', 'xgb', 'rnn', 'lstm']

BASE_FEATURES = ['NO', 'NO2', 'NOx', 'Veloc.', 'Temp.', 'R.Sol.']
CATEGORICAL = ['Estacion', 'Transecto']
CIRCULAR_ORIG = ['Direc.', 'Angulo']
TARGET = 'O3'

# ---------------------- FUNCIONES AUXILIARES ----------------------

def ensure_datetime_index(df: pd.DataFrame, col: str = 'datetime') -> pd.DataFrame:
    """Asegura DatetimeIndex con frecuencia horaria y reindexa al rango completo."""
    df = df.copy()
    if col in df.columns:
        # parseamos si no está datetime
        df[col] = pd.to_datetime(df[col], errors='coerce')
        df = df.set_index(col)
    else:
        if not isinstance(df.index, pd.DatetimeIndex):
            raise ValueError('No hay columna datetime ni index datetime.')
    df = df.sort_index()
    if df.empty:
        raise ValueError('El DataFrame está vacío después de convertir datetime.')
    full_idx = pd.date_range(df.index.min(), df.index.max(), freq='h')
    df = df.reindex(full_idx)
    df.index.name = 'datetime'
    return df


def add_cyclical_datetime_features(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    idx = d.index
    hour = idx.hour
    day = idx.day
    # isocalendar may return a DataFrame in some pandas versions:
    try:
        week = idx.isocalendar().week.astype(int)
    except Exception:
        week = pd.Index([i.isocalendar()[1] for i in idx]).astype(int)
    month = idx.month
    year = idx.year

    d['hour_sin'] = np.sin(2 * np.pi * hour / 24)
    d['hour_cos'] = np.cos(2 * np.pi * hour / 24)
    d['day_sin'] = np.sin(2 * np.pi * (day - 1) / 31)
    d['day_cos'] = np.cos(2 * np.pi * (day - 1) / 31)
    d['week_sin'] = np.sin(2 * np.pi * (week - 1) / 52)
    d['week_cos'] = np.cos(2 * np.pi * (week - 1) / 52)
    d['month_sin'] = np.sin(2 * np.pi * (month - 1) / 12)
    d['month_cos'] = np.cos(2 * np.pi * (month - 1) / 12)
    years = year - year.min()
    span = (year.max() - year.min() + 1) if (year.max() - year.min() + 1) > 0 else 1
    d['year_sin'] = np.sin(2 * np.pi * years / span)
    d['year_cos'] = np.cos(2 * np.pi * years / span)
    return d


def circular_to_sincos(series: pd.Series, prefix: str) -> pd.DataFrame:
    radians = np.deg2rad(series.astype(float))
    return pd.DataFrame({f'{prefix}_sin': np.sin(radians), f'{prefix}_cos': np.cos(radians)}, index=series.index)


def choose_imputation_strategy(series: pd.Series, var_name: str) -> str:
    if var_name in IMPUTE_PREFS:
        return IMPUTE_PREFS[var_name]

    n = len(series)
    missing = int(series.isna().sum())
    frac = missing / n if n > 0 else 1.0

    if var_name in CATEGORICAL:
        return 'ffill'
    if var_name in CIRCULAR_ORIG:
        return 'circular'

    if frac < MISSING_THRESHOLDS['low']:
        return 'time_interpolate'
    if frac < MISSING_THRESHOLDS['medium']:
        return 'knn'
    if frac < MISSING_THRESHOLDS['high']:
        return 'iterative'
    return 'median'


def impute_variable_adaptive(df: pd.DataFrame, col: str) -> pd.Series:
    strat = choose_imputation_strategy(df[col], col)
    if strat == 'time_interpolate':
        s_imputed = df[col].interpolate(method='time', limit_direction='both')
        if s_imputed.isna().any():
            s_imputed = s_imputed.fillna(s_imputed.median())
        return s_imputed
    elif strat == 'knn':
        numeric_cols = [c for c in df.columns if df[c].dtype.kind in 'bifc']
        if len(numeric_cols) == 0:
            return df[col].fillna(df[col].median())
        knn = KNNImputer(n_neighbors=5)
        arr = knn.fit_transform(df[numeric_cols])
        res = pd.DataFrame(arr, index=df.index, columns=numeric_cols)[col]
        return res
    elif strat == 'iterative':
        numeric_cols = [c for c in df.columns if df[c].dtype.kind in 'bifc']
        if len(numeric_cols) == 0:
            return df[col].fillna(df[col].median())
        imp = IterativeImputer(max_iter=10, random_state=0)
        arr = imp.fit_transform(df[numeric_cols])
        res = pd.DataFrame(arr, index=df.index, columns=numeric_cols)[col]
        return res
    elif strat == 'median':
        return df[col].fillna(df[col].median())
    elif strat == 'ffill':
        return df[col].fillna(method='ffill').fillna(method='bfill')
    elif strat == 'circular':
        raise ValueError('La imputación circular debe manejarse fuera de esta función sobre sin/cos')
    else:
        raise ValueError(f'Estrategia desconocida: {strat}')


# ---------------------- PROCESAMIENTO DE UN CSV ----------------------

def preprocess_single_csv(path: str) -> Tuple[pd.DataFrame, Dict]:
    """
    Carga un CSV, genera features cíclicas, codifica circularmente Direc./Angulo y aplica imputación adaptativa.
    Devuelve el dataframe procesado y metadatos (mappings para categorías).
    """
    try:
        df = pd.read_csv(path)
    except Exception as e:
        raise RuntimeError(f'Error leyendo {path}: {e}')

    df.columns = [c.strip() for c in df.columns]

    df = ensure_datetime_index(df, 'datetime')

    # generar features temporales cíclicas
    df = add_cyclical_datetime_features(df)

    # Manejar columnas circulares: Direc. y Angulo -> sin/cos (prefijo limpio)
    for circ_col in CIRCULAR_ORIG:
        if circ_col in df.columns:
            pref = circ_col.replace('.', '').replace(' ', '_')
            sincos = circular_to_sincos(df[circ_col], pref)
            df = pd.concat([df, sincos], axis=1)

    # imputar componentes circulares primero (si corresponde)
    for circ_col in CIRCULAR_ORIG:
        pref = circ_col.replace('.', '').replace(' ', '_')
        sin_col = f'{pref}_sin'
        cos_col = f'{pref}_cos'
        if sin_col in df.columns and cos_col in df.columns:
            df[sin_col] = df[sin_col].interpolate(method='time', limit_direction='both')
            df[cos_col] = df[cos_col].interpolate(method='time', limit_direction='both')
            if df[sin_col].isna().any():
                df[sin_col] = df[sin_col].fillna(df[sin_col].median())
            if df[cos_col].isna().any():
                df[cos_col] = df[cos_col].fillna(df[cos_col].median())

    # ahora imputación adaptativa para el resto de columnas
    # iteramos sobre columnas (lista en orden estable)
    for col in list(df.columns):
        # Saltar columnas originales circulares (Direc., Angulo) porque ya imputamos sus sin/cos
        if col in CIRCULAR_ORIG:
            continue
        # Saltar las componentes sin/cos asociadas a las variables circulares (ya imputadas)
        if any(col == (c.replace('.', '').replace(' ', '_') + suffix) for c in CIRCULAR_ORIG for suffix in ['_sin', '_cos']):
            continue
        # Las features cíclicas derivadas del índice no deberían tener NaNs
        if col in ['hour_sin', 'hour_cos', 'day_sin', 'day_cos', 'week_sin', 'week_cos', 'month_sin', 'month_cos', 'year_sin', 'year_cos']:
            continue
        # Columnas categóricas simples: rellenar con forward/back fill
        if col in ['Estacion', 'Transecto', 'Dist.']:
            if col in df.columns:
                df[col] = df[col].fillna(method='ffill').fillna(method='bfill')
            continue
        # Si columna existe y es numérica -> imputación adaptativa
        if col in df.columns and df[col].dtype.kind in 'bifc':
            df[col] = impute_variable_adaptive(df, col)
        else:
            # columnas no numéricas: rellenar con forward/back
            if col in df.columns:
                df[col] = df[col].fillna(method='ffill').fillna(method='bfill')

    # asegurar que no haya NaNs en las columnas base y target (por seguridad)
    for col in BASE_FEATURES + [TARGET]:
        if col in df.columns and df[col].isna().any():
            df[col] = df[col].fillna(df[col].median())

    return df, {'columns': df.columns.tolist()}


# ---------------------- CREAR VENTANAS SUPERVISADAS ----------------------

def make_sliding_windows(df: pd.DataFrame,
                         input_hours: int = INPUT_HOURS,
                         output_hours: int = OUTPUT_HOURS,
                         features: List[str] = None,
                         target_col: str = TARGET,
                         model_type: str = 'rf'):
    """
    Crea muestras deslizantes.
    """
    if features is None:
        features = BASE_FEATURES + [c for c in df.columns if c.startswith('hour_') or c.endswith('_sin') or c.endswith('_cos')]
    features = [f for f in features if f in df.columns]

    X_list = []
    y_list = []
    idxs = []
    total_len = len(df)
    for start in range(0, total_len - input_hours - output_hours + 1):
        end_input = start + input_hours
        end_output = end_input + output_hours
        Xw = df.iloc[start:end_input][features].values
        yw = df.iloc[end_input:end_output][target_col].values if target_col in df.columns else np.full((output_hours,), np.nan)
        if np.isnan(Xw).any() or np.isnan(yw).any():
            continue
        X_list.append(Xw)
        y_list.append(yw)
        idxs.append(df.index[end_input - 1])

    if model_type in ['rf', 'xgb']:
        X_flat = np.array(X_list)
        if X_flat.size == 0:
            return pd.DataFrame(), pd.DataFrame()
        n_samples, t, n_feat = X_flat.shape
        X_2d = X_flat.reshape(n_samples, t * n_feat)
        y_2d = np.array(y_list)
        y_cols = [f'y_{i+1}' for i in range(y_2d.shape[1])]
        X_df = pd.DataFrame(X_2d, index=idxs)
        y_df = pd.DataFrame(y_2d, columns=y_cols, index=idxs)
        return X_df, y_df
    else:
        if len(X_list) == 0:
            return np.empty((0, input_hours, len(features))), np.empty((0, output_hours))
        X_arr = np.array(X_list)
        y_arr = np.array(y_list)
        return X_arr, y_arr


# ---------------------- CREAR DATASETS POR ESTACION Y COMBINADO ----------------------

def prepare_and_save_all(file_paths: List[str], output_dir: Path):
    # compatibilidad para versiones de sklearn: sparse_output (nueva) vs sparse (antigua)
    try:
        global_onehot = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    except TypeError:
        global_onehot = OneHotEncoder(sparse=False, handle_unknown='ignore')

    processed = {}
    for p in file_paths:
        name = Path(p).stem
        print(f'Procesando {name}...')
        try:
            df, meta = preprocess_single_csv(p)
        except Exception as e:
            print(f'  ERROR procesando {name}: {e}  -> se omite este archivo.')
            continue
        processed[name] = df

    if len(processed) == 0:
        raise RuntimeError('No se procesó ninguna estación correctamente. Revisa las rutas o los CSVs.')

    # FIT global onehot sobre todas las estaciones/ transectos
    all_cats = pd.DataFrame({
        'Estacion': [df['Estacion'].iloc[0] if 'Estacion' in df.columns and not df['Estacion'].isna().all() else name for name, df in processed.items()],
        'Transecto': [df['Transecto'].iloc[0] if 'Transecto' in df.columns and not df['Transecto'].isna().all() else name for name, df in processed.items()]
    }).astype(str)
    global_onehot.fit(all_cats)
    joblib.dump(global_onehot, output_dir / 'global_onehot_encoder.joblib')

    # 1) Dataset por estación (4 por estación)
    for name, df in processed.items():
        station_dir = output_dir / 'per_station' / name
        station_dir.mkdir(parents=True, exist_ok=True)

        # RF/XGB: one-hot por estación local (opcional)
        df_tab = df.copy()
        cats = [c for c in CATEGORICAL if c in df_tab.columns]
        if cats:
            df_tab = pd.get_dummies(df_tab, columns=cats, dummy_na=False)

        X_rf, y_rf = make_sliding_windows(df_tab, model_type='rf')
        if not X_rf.empty:
            X_rf.to_csv(station_dir / 'rf_X.csv', index=True)
            y_rf.to_csv(station_dir / 'rf_y.csv', index=True)
        else:
            print(f'Advertencia: no se generaron ventanas RF para {name} (poca longitud)')

        X_xgb, y_xgb = make_sliding_windows(df_tab, model_type='xgb')
        if not X_xgb.empty:
            X_xgb.to_csv(station_dir / 'xgb_X.csv', index=True)
            y_xgb.to_csv(station_dir / 'xgb_y.csv', index=True)

        # RNN/LSTM: label encoding per-estación
        df_seq = df.copy()
        encoders = {}
        for cat in CATEGORICAL:
            if cat in df_seq.columns:
                le = LabelEncoder()
                # convertir a str para evitar problemas con NaN
                df_seq[cat + '_id'] = le.fit_transform(df_seq[cat].astype(str))
                encoders[cat] = le
        joblib.dump(encoders, station_dir / 'label_encoders.joblib')

        seq_features = [f for f in df_seq.columns if (
            f in BASE_FEATURES or f.endswith('_sin') or f.endswith('_cos') or f.endswith('_id') or f.startswith('hour_') or f.startswith('day_') or f.startswith('week_') or f.startswith('month_') or f.startswith('year_')
        )]
        X_rnn, y_rnn = make_sliding_windows(df_seq, features=seq_features, model_type='rnn')
        np.savez_compressed(station_dir / 'rnn_data.npz', X=X_rnn, y=y_rnn)
        np.savez_compressed(station_dir / 'lstm_data.npz', X=X_rnn, y=y_rnn)

    # 2) Dataset combinado (todas las estaciones en uno)
    combined = pd.concat(processed.values(), keys=processed.keys(), names=['station', 'datetime'])
    # resetear la primera nivel ('station') a columna y mantener el índice datetime
    combined = combined.reset_index(level=0).rename(columns={'station': 'Estacion_source'})

    combined_dir = output_dir / 'combined'
    combined_dir.mkdir(parents=True, exist_ok=True)

    # Asegurar columnas Estacion/Transecto en combined
    for cat in CATEGORICAL:
        if cat not in combined.columns:
            combined[cat] = combined['Estacion_source']

    # Cargar encoder global y transformar
    oh = joblib.load(output_dir / 'global_onehot_encoder.joblib')
    cat_df = combined[['Estacion', 'Transecto']].fillna('unknown').astype(str)
    oh_arr = oh.transform(cat_df)
    try:
        oh_cols = oh.get_feature_names_out(['Estacion', 'Transecto'])
    except Exception:
        # fallback a nombres genéricos si versión antigua
        oh_cols = [f'{col}_{val}' for col, vals in zip(['Estacion', 'Transecto'], [all_cats['Estacion'].unique(), all_cats['Transecto'].unique()]) for val in vals]

    oh_df = pd.DataFrame(oh_arr, index=combined.index, columns=oh_cols)
    combined_tab = pd.concat([combined.drop(columns=CATEGORICAL, errors='ignore'), oh_df], axis=1)

    # crear datasets combinados para RF/XGB
    X_rf_c, y_rf_c = make_sliding_windows(combined_tab, model_type='rf')
    if not (isinstance(X_rf_c, pd.DataFrame) and X_rf_c.empty):
        X_rf_c.to_csv(combined_dir / 'rf_X.csv', index=True)
        y_rf_c.to_csv(combined_dir / 'rf_y.csv', index=True)
    else:
        print('Advertencia: no se generaron ventanas RF para el dataset combinado.')

    X_xgb_c, y_xgb_c = make_sliding_windows(combined_tab, model_type='xgb')
    if not (isinstance(X_xgb_c, pd.DataFrame) and X_xgb_c.empty):
        X_xgb_c.to_csv(combined_dir / 'xgb_X.csv', index=True)
        y_xgb_c.to_csv(combined_dir / 'xgb_y.csv', index=True)

    # Para RNN/LSTM en combinado: label encoding global para Estacion/Transecto
    enc_global = {}
    for cat in CATEGORICAL:
        le = LabelEncoder()
        combined[cat + '_id'] = le.fit_transform(combined[cat].astype(str))
        enc_global[cat] = le
    joblib.dump(enc_global, combined_dir / 'global_label_encoders.joblib')

    seq_features_comb = [f for f in combined.columns if (
        f in BASE_FEATURES or f.endswith('_sin') or f.endswith('_cos') or f.endswith('_id') or f.startswith('hour_') or f.startswith('day_') or f.startswith('week_') or f.startswith('month_') or f.startswith('year_')
    )]
    X_rnn_c, y_rnn_c = make_sliding_windows(combined, features=seq_features_comb, model_type='rnn')
    np.savez_compressed(combined_dir / 'rnn_data.npz', X=X_rnn_c, y=y_rnn_c)
    np.savez_compressed(combined_dir / 'lstm_data.npz', X=X_rnn_c, y=y_rnn_c)

    print('Datasets generados y guardados en:', output_dir)


if __name__ == '__main__':
    prepare_and_save_all(FILE_PATHS, OUTPUT_DIR)

Procesando T1_E1_Alicante...


/var/folders/85/2m2sr7rx6g5d561zn04dcq000000gn/T/ipykernel_20013/3460071549.py:228: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[col] = df[col].fillna(method='ffill').fillna(method='bfill')


Procesando T1_E2_Elda...
